In [1]:
from GCMC import *
import pandas as pd
import psycopg2
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
# Connect PostgreSQL
conn = psycopg2.connect(
    dbname="mydb",
    user="user",
    password="pass",
    host="my_postgres",
    port="5432"
)

# Import data using Pandas
df = pd.read_sql(
    """SELECT index
            , recipe_code
            , recipe_name
            , user_id
            , stars
            , user_reputation
            , food_category
            , feature 
       FROM new_review""", conn)
print(df.head())

conn.close()


   index  recipe_code         recipe_name         user_id  stars  \
0      0        14299  Creamy White Chili  u_9iFLIhMa8QaG      5   
1      1        14299  Creamy White Chili  u_Lu6p25tmE77j      5   
2      2        14299  Creamy White Chili  u_s0LwgpZ8Jsqq      5   
3      3        14299  Creamy White Chili  u_fqrybAdYjgjG      0   
4      4        14299  Creamy White Chili  u_XXWKwVhKZD69      0   

   user_reputation food_category                           feature  
0                1    Soup/Chili  #creamy, #white, #chili, #hearty  
1               50    Soup/Chili  #creamy, #white, #chili, #hearty  
2               10    Soup/Chili  #creamy, #white, #chili, #hearty  
3                1    Soup/Chili  #creamy, #white, #chili, #hearty  
4               10    Soup/Chili  #creamy, #white, #chili, #hearty  


/tmp/ipykernel_37655/3542586661.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [ ]:
def split_data(df):
    
    total_indices = np.arange(len(df))
    np.random.shuffle(total_indices)

    train_size = int(len(total_indices) * 0.7)
    valid_size = int(len(total_indices) * 0.15)  # validation 15%
    test_size = len(total_indices) - train_size - valid_size  # 나머지는 test

    train_indices = total_indices[:train_size]
    valid_indices = total_indices[train_size:train_size + valid_size]
    test_indices = total_indices[train_size + valid_size:]
    
    train_df = df.iloc[train_indices]
    test_df = df.iloc[test_indices]
    valid_df = df.iloc[valid_indices] 
    
    return train_df, test_df, valid_df

def safe_transform(encoder, values, unknown_val=-1):
    known = set(encoder.classes_)
    class_to_index = {cls: i for i, cls in enumerate(encoder.classes_)}
    return [class_to_index[v] if v in known else unknown_val for v in values]
    
def preprocessing(df):
    
    global num_users, num_items
    
    user_encoder = LabelEncoder().fit(df['user_id'])
    item_encoder = LabelEncoder().fit(df['recipe_code'])

    df['user_idx'] = user_encoder.transform(df['user_id'])
    df['item_idx'] = item_encoder.transform(df['recipe_code'])

    train_df, test_df, valid_df = split_data(df) 
    
    num_users = len(user_encoder.classes_)
    num_items = len(item_encoder.classes_)
    
    # train
    user_reputation_train = train_df.groupby('user_idx')['user_reputation'].mean().reindex(range(num_users)).fillna(0)
    u_feat_side_train = torch.tensor(user_reputation_train.values).unsqueeze(1)  # shape: [num_users, 1]
    u_feat_side_train = u_feat_side_train.to(torch.float32) 
    
    # test
    user_reputation_test = test_df.groupby('user_idx')['user_reputation'].mean().reindex(range(num_users)).fillna(0)
    u_feat_side_test = torch.tensor(user_reputation_test.values).unsqueeze(1)  # shape: [num_users, 1]
    u_feat_side_test = u_feat_side_test.to(torch.float32)
     
    cat_encoder = OneHotEncoder()
        
    mlb = MultiLabelBinarizer()
  
    # create category side info based on total items
    cat_onehot = cat_encoder.fit_transform(df[['food_category']])
    item_cat = pd.DataFrame(cat_onehot.toarray()).groupby(df['item_idx']).mean()
    item_cat = item_cat.reindex(range(num_items)).fillna(0)
    v_cat_side = torch.tensor(item_cat.values, dtype=torch.float32)

    # binarize feature list based on total items
    df['feature_list'] = df['feature'].fillna("").apply(lambda x: x.split(', ') if x else [])
    tag_binary = mlb.fit_transform(df['feature_list'])
    item_feat = pd.DataFrame(tag_binary).groupby(df['item_idx']).mean()
    item_feat = item_feat.reindex(range(num_items)).fillna(0)
    v_tag_side = torch.tensor(item_feat.values, dtype=torch.float32)

    # final item side info
    v_feat_side = torch.cat([v_cat_side, v_tag_side], dim=1)
    v_feat_side = v_feat_side.to(torch.float32)

    
    return num_users, num_items, u_feat_side_train, v_feat_side, u_feat_side_test, train_df, test_df, valid_df 





In [4]:
# list of sparse adjacency matrix, indicating user_item relationship by stars
# support[i] indicates user-
def make_support_matrix(df, num_users, num_items, num_classes):
    supports = []
    for rating in range(1, num_classes + 1):
        mask = df['stars'] == rating
        rows = df[mask]['user_idx'].values
        cols = df[mask]['item_idx'].values
        values = np.ones(len(rows))

        coo = torch.sparse_coo_tensor(
            indices=torch.tensor([rows, cols]),
            values=torch.tensor(values, dtype=torch.float32),
            size=(num_users, num_items)
        )
        supports.append(coo.coalesce())
    return supports


In [ ]:
num_users, num_items, u_feat_side_train, v_feat_side, u_feat_side_test, train_df, test_df, valid_df = preprocessing(df)

In [6]:
support = make_support_matrix(train_df, num_users, num_items, 6)
support_t = [s.transpose(0, 1) for s in support]

/tmp/ipykernel_37655/3047605791.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  indices=torch.tensor([rows, cols]),


In [ ]:
num_epochs = 50


model = RecommenderSideInfoGAE(
    input_dim=128,           
    feat_hidden_dim= u_feat_side_train.shape[1],        # u_feat_side.shape[1], v_feat_side.shape[1]
    hidden_dims=[64, 32],       # freely setting
    num_support = len(support),
    num_classes=6,              # The number of the star's class
    num_basis_functions=3,      # The number of decoder basis 
    num_users=num_users,        # The number of users based on LabelEncoder
    num_items=num_items,        # The number of items based on LabelEncoder
    u_num_side_features=u_feat_side_train.shape[1],  # Demension of item's side info
    v_num_side_features=v_feat_side.shape[1],
    accum='sum',                # 'sum' or 'stack'
    self_connections=False,     # including self-loops in the GCN
    dropout=0.5                 # percentage of dropout 
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

best_loss = float('inf')
best_model_state = None

for epoch in range(num_epochs):
    model.train()
    
    # After shuffling, divide batch
    shuffled = train_df.sample(frac=1).reset_index(drop=True)
    batch_size = 512

    for i in range(0, len(shuffled), batch_size):
        batch_df = shuffled.iloc[i:i+batch_size]

        u_idx = torch.tensor(batch_df['user_idx'].values, dtype=torch.long)
        v_idx = torch.tensor(batch_df['item_idx'].values, dtype=torch.long)        
        labels = torch.tensor(batch_df['stars'].values, dtype=torch.long)

        output = model(
            u_feat_side_train, v_feat_side,
            support, support_t,
            u_idx, v_idx
        )

        loss = F.nll_loss(output, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_model_state = model.state_dict()  # save the current model's weight

    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")
    
print(f"Best model's loss is {best_loss}")
torch.save(best_model_state, "/root/profile/tech blog/model_weight/best_model.pth")


Epoch 1 | Loss: 2.3672
Epoch 2 | Loss: 1.6195
Epoch 3 | Loss: 1.5970
Epoch 4 | Loss: 1.1588
Epoch 5 | Loss: 1.3566
Epoch 6 | Loss: 1.0869
Epoch 7 | Loss: 0.9971
Epoch 8 | Loss: 1.0691
Epoch 9 | Loss: 0.9737
Epoch 10 | Loss: 1.0914
Epoch 11 | Loss: 1.0325
Epoch 12 | Loss: 0.9869
Epoch 13 | Loss: 0.8598
Epoch 14 | Loss: 1.0253
Epoch 15 | Loss: 0.9298
Epoch 16 | Loss: 1.0774
Epoch 17 | Loss: 0.8024
Epoch 18 | Loss: 0.9072
Epoch 19 | Loss: 0.8037
Epoch 20 | Loss: 0.9722
Epoch 21 | Loss: 0.7642
Epoch 22 | Loss: 0.8398
Epoch 23 | Loss: 0.8462
Epoch 24 | Loss: 0.7888
Epoch 25 | Loss: 0.7529
Epoch 26 | Loss: 0.8292
Epoch 27 | Loss: 0.7716
Epoch 28 | Loss: 0.8131
Epoch 29 | Loss: 0.7179
Epoch 30 | Loss: 0.7917
Epoch 31 | Loss: 0.7157
Epoch 32 | Loss: 0.8098
Epoch 33 | Loss: 0.8321
Epoch 34 | Loss: 0.7986
Epoch 35 | Loss: 0.7402
Epoch 36 | Loss: 0.6617
Epoch 37 | Loss: 0.8072
Epoch 38 | Loss: 0.8255
Epoch 39 | Loss: 0.7103
Epoch 40 | Loss: 0.7255
Epoch 41 | Loss: 0.7783
Epoch 42 | Loss: 0.6332
E

In [8]:
u_indices_test = torch.tensor(test_df['user_idx'].values, dtype=torch.long)
v_indices_test = torch.tensor(test_df['item_idx'].values, dtype=torch.long)
labels_test = torch.tensor(test_df['stars'].values , dtype=torch.long)

# forward
output = model(
    u_feat_side_test, v_feat_side,
    support, support_t,
    u_indices_test, v_indices_test
)

loss = F.nll_loss(output, labels_test)


In [9]:
loss

tensor(2.8630, grad_fn=<NllLossBackward0>)

In [24]:
torch.where(output[0] == max(output[0]))[0].item()

5

In [25]:
result = []
for l in output:
    result.append(torch.where(l == max(l))[0].item())

In [ ]:
test_df['expectation'] = result
test_df['con_yn'] = test_df.stars == test_df.expectation

/tmp/ipykernel_37655/45162741.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['expectation'] = result


In [33]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_df['stars'], test_df['expectation'])

In [34]:
cm

array([[ 215,    3,    0,    0,    1,   29],
       [  38,    0,    0,    0,    0,    2],
       [  26,    0,    0,    0,    0,    7],
       [  61,    0,    0,    0,    0,   22],
       [ 181,    3,    0,    0,    0,   78],
       [1415,   12,    6,    0,    5,  624]])